# BERT Sentiment Analysis with Token-Level Integrated Gradients

**Learning objectives:**
- Apply LayerIntegratedGradients to a pretrained BERT sentiment classifier
- Visualize token attributions with colored text (green=positive, red=negative)
- Handle subword tokenization: aggregate subword attributions to word level
- Compute convergence delta to verify attribution quality
- Compare attributions for positive and negative sentences

**Estimated time:** 15 minutes

**Model:** `textattack/bert-base-uncased-SST-2` (HuggingFace)

**Dataset:** Stanford Sentiment Treebank (SST-2) sentences

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerIntegratedGradients

print("Imports complete.")

# Load pretrained BERT fine-tuned on SST-2
model_name = "textattack/bert-base-uncased-SST-2"
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

LABELS = ["NEGATIVE", "POSITIVE"]
print(f"Model loaded. Labels: {LABELS}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 1. Define Helper Functions

In [ ]:
def forward_func(input_ids, attention_mask=None, token_type_ids=None):
    """Forward pass returning logits for target attribution."""
    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
    )
    return output.logits


def tokenize_and_encode(text: str, max_length: int = 128):
    """Tokenize text and return all tensors needed for attribution."""
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=max_length,
        truncation=True,
        padding=False,
    )
    return (
        inputs['input_ids'],
        inputs.get('attention_mask'),
        inputs.get('token_type_ids'),
    )


def make_pad_baseline(input_ids: torch.Tensor) -> torch.Tensor:
    """Create all-PAD baseline of the same shape as input_ids."""
    return torch.full_like(input_ids, tokenizer.pad_token_id)


def predict(text: str):
    """Get prediction and confidence for a text."""
    ids, mask, ttype = tokenize_and_encode(text)
    with torch.no_grad():
        logits = forward_func(ids, mask, ttype)
    probs = torch.softmax(logits, dim=1)[0]
    pred_class = probs.argmax().item()
    return pred_class, probs[pred_class].item(), LABELS[pred_class]


# Test prediction
test_text = "This film is an absolute masterpiece with stunning performances."
cls, conf, label = predict(test_text)
print(f"Text:       '{test_text}'")
print(f"Prediction: {label} ({conf:.1%} confidence)")

## 2. Compute Token Attributions with LayerIntegratedGradients

In [ ]:
def compute_token_attributions(text: str, n_steps: int = 50):
    """Compute LayerIG token attributions for a text.
    
    Returns:
        tokens: list of token strings
        attrs: numpy array of token attribution scores
        pred_class: predicted class index
        pred_label: predicted class label
        confidence: prediction confidence
        delta: convergence delta
    """
    input_ids, attention_mask, token_type_ids = tokenize_and_encode(text)
    baseline_ids = make_pad_baseline(input_ids)

    # Get prediction
    with torch.no_grad():
        logits = forward_func(input_ids, attention_mask, token_type_ids)
    probs = torch.softmax(logits, dim=1)[0]
    pred_class = probs.argmax().item()

    # LayerIG at BERT's embedding layer
    lig = LayerIntegratedGradients(
        forward_func=forward_func,
        layer=model.bert.embeddings,
    )

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask, token_type_ids),
        target=pred_class,
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    # Sum over embedding dimension → one score per token
    token_scores = attributions.sum(dim=-1).squeeze(0).detach().numpy()

    # Token strings
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    return {
        'tokens': tokens,
        'attrs': token_scores,
        'pred_class': pred_class,
        'pred_label': LABELS[pred_class],
        'confidence': probs[pred_class].item(),
        'delta': delta.item(),
        'input_ids': input_ids,
    }


# Test
result = compute_token_attributions(test_text)
print(f"Prediction: {result['pred_label']} ({result['confidence']:.1%})")
print(f"Convergence delta: {result['delta']:.6f}")
print(f"Tokens: {result['tokens']}")
print(f"Attributions: {result['attrs'].round(4)}")

## 3. Subword Aggregation: Merge WordPiece Tokens to Words

In [ ]:
def aggregate_subwords(tokens: list, attrs: np.ndarray, skip_special: bool = True):
    """Merge WordPiece subword tokens back to words and sum their attributions.
    
    Special tokens ([CLS], [SEP], [PAD]) are optionally excluded.
    Tokens starting with '##' are continuations of the previous word.
    """
    special_tokens = {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<pad>'}
    word_tokens = []
    word_attrs  = []
    current_word = None
    current_attr = 0.0

    for tok, attr in zip(tokens, attrs):
        if skip_special and tok in special_tokens:
            continue
        if tok.startswith('##'):
            # Continuation subword
            if current_word is not None:
                current_word += tok[2:]
                current_attr += attr
            else:
                # Edge case: ## without preceding token
                current_word = tok[2:]
                current_attr = attr
        else:
            # New word
            if current_word is not None:
                word_tokens.append(current_word)
                word_attrs.append(current_attr)
            current_word = tok
            current_attr = attr

    # Don't forget the last word
    if current_word is not None:
        word_tokens.append(current_word)
        word_attrs.append(current_attr)

    return word_tokens, np.array(word_attrs)


word_tokens, word_attrs = aggregate_subwords(result['tokens'], result['attrs'])
print("Word-level attributions:")
for word, attr in zip(word_tokens, word_attrs):
    bar = '█' * int(abs(attr) / max(abs(word_attrs)) * 20)
    sign = '+' if attr > 0 else '-'
    print(f"  {word:<20} {sign}{abs(attr):.4f}  {bar}")

## 4. Colored Text Visualization

In [ ]:
def visualize_token_attributions(word_tokens: list, word_attrs: np.ndarray,
                                   title: str, pred_label: str, confidence: float,
                                   figsize: tuple = None):
    """Create colored text visualization of token attributions."""
    n_words = len(word_tokens)
    if figsize is None:
        figsize = (max(10, n_words * 1.1), 3.0)

    # Normalize
    max_abs = np.abs(word_attrs).max()
    attrs_norm = word_attrs / max_abs if max_abs > 0 else word_attrs

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')

    token_w = 1.0 / max(n_words, 1)

    for i, (word, a, a_raw) in enumerate(zip(word_tokens, attrs_norm, word_attrs)):
        x_center = (i + 0.5) / n_words

        # Color mapping
        if a > 0:  # Green for positive
            r = 1 - a * 0.6
            g = 1.0
            b = 1 - a * 0.6
        else:       # Red for negative
            r = 1.0
            g = 1 + a * 0.6
            b = 1 + a * 0.6
        bg_color = (np.clip(r, 0, 1), np.clip(g, 0, 1), np.clip(b, 0, 1))

        # Box
        rect = plt.Rectangle(
            (x_center - token_w*0.45, 0.25),
            token_w * 0.9, 0.55,
            facecolor=bg_color, edgecolor='#cccccc', linewidth=0.5,
        )
        ax.add_patch(rect)

        # Word text
        font_size = max(7, min(12, 100 // n_words))
        ax.text(x_center, 0.53, word,
                ha='center', va='center', fontsize=font_size, color='black')

        # Attribution value
        ax.text(x_center, 0.17, f"{a_raw:+.3f}",
                ha='center', va='center', fontsize=6, color='#555555')

    # Legend
    pos_patch = mpatches.Patch(facecolor='#66cc66', label='Positive attribution')
    neg_patch = mpatches.Patch(facecolor='#ff6666', label='Negative attribution')
    neu_patch = mpatches.Patch(facecolor='white', edgecolor='#cccccc', label='Neutral')
    ax.legend(handles=[pos_patch, neg_patch, neu_patch],
              loc='upper right', fontsize=8, framealpha=0.9)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    pred_color = '#2ca25f' if pred_label == 'POSITIVE' else '#d73027'
    ax.set_title(
        f"{title}\nPrediction: "
        + r"$\bf{" + pred_label + r"}$"
        + f" ({confidence:.1%} confidence)",
        fontsize=12, pad=10, color=pred_color
    )
    return fig


# Visualize
fig = visualize_token_attributions(
    word_tokens, word_attrs,
    title=f'"{test_text}"',
    pred_label=result['pred_label'],
    confidence=result['confidence'],
)
plt.tight_layout()
plt.savefig('bert_ig_positive.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_ig_positive.png")

## 5. Attribution Gallery: Multiple Sentences

In [ ]:
# Test sentences: positive, negative, and tricky cases
test_sentences = [
    "This film is an absolute masterpiece with stunning performances.",
    "The acting was terrible and the plot made no sense whatsoever.",
    "The movie was not boring at all, surprisingly engaging throughout.",  # negation
    "A disappointingly average film with some bright moments.",  # mixed
    "I did not dislike the film.",  # double negation
]

print("Computing attributions for all test sentences...")
all_results = []
for text in test_sentences:
    result = compute_token_attributions(text, n_steps=40)
    word_tokens, word_attrs = aggregate_subwords(result['tokens'], result['attrs'])
    result['word_tokens'] = word_tokens
    result['word_attrs']  = word_attrs
    all_results.append(result)
    print(f"  '{text[:50]}...' → {result['pred_label']} ({result['confidence']:.1%}) δ={result['delta']:.4f}")

print("Done.")

In [ ]:
# Create a multi-row figure
fig, axes = plt.subplots(len(test_sentences), 1, figsize=(18, 3.5 * len(test_sentences)))

for ax, result in zip(axes, all_results):
    word_tokens = result['word_tokens']
    word_attrs  = result['word_attrs']
    n_words = len(word_tokens)

    ax.axis('off')
    max_abs = np.abs(word_attrs).max() if np.abs(word_attrs).max() > 0 else 1
    attrs_norm = word_attrs / max_abs

    for i, (word, a, a_raw) in enumerate(zip(word_tokens, attrs_norm, word_attrs)):
        x_center = (i + 0.5) / n_words
        token_w = 1.0 / max(n_words, 1)

        if a > 0:
            r, g, b = 1 - a*0.6, 1.0, 1 - a*0.6
        else:
            r, g, b = 1.0, 1+a*0.6, 1+a*0.6
        bg_color = (np.clip(r, 0, 1), np.clip(g, 0, 1), np.clip(b, 0, 1))

        ax.add_patch(plt.Rectangle(
            (x_center - token_w*0.44, 0.2), token_w*0.88, 0.65,
            facecolor=bg_color, edgecolor='#dddddd', linewidth=0.4
        ))
        ax.text(x_center, 0.55, word, ha='center', va='center',
                fontsize=max(7, min(11, 80//n_words)), color='black')
        ax.text(x_center, 0.12, f"{a_raw:+.3f}",
                ha='center', va='center', fontsize=5.5, color='#666666')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    pred_color = '#2ca25f' if result['pred_label'] == 'POSITIVE' else '#d73027'
    ax.set_title(
        f"{result['pred_label']} ({result['confidence']:.1%}) | δ={result['delta']:.3f}",
        fontsize=9, color=pred_color, pad=3
    )

plt.suptitle('BERT SST-2: LayerIntegratedGradients Token Attribution Gallery\n'
             '(Green=toward predicted class, Red=against predicted class)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('bert_ig_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_ig_gallery.png")

## 6. Convergence Analysis: How Many Steps Are Enough?

In [ ]:
# How does convergence delta improve with more integration steps?
sample_text = test_sentences[0]
ids, mask, ttype = tokenize_and_encode(sample_text)
baseline_ids = make_pad_baseline(ids)

with torch.no_grad():
    pred_class = forward_func(ids, mask, ttype).argmax(dim=1).item()

lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

steps_range = [10, 20, 30, 50, 100]
deltas = []

for n_steps in steps_range:
    _, delta = lig.attribute(
        inputs=ids,
        baselines=baseline_ids,
        additional_forward_args=(mask, ttype),
        target=pred_class,
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    deltas.append(abs(delta.item()))
    print(f"  n_steps={n_steps:4d}: |δ| = {abs(delta.item()):.6f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(steps_range, deltas, 'o-', color='#377eb8', linewidth=2, markersize=8)
ax.axhline(0.01, color='#4daf4a', linestyle='--', label='Acceptable δ < 0.01')
ax.axhline(0.001, color='#e41a1c', linestyle='--', label='Good δ < 0.001')
ax.set_xlabel('Integration steps (n_steps)')
ax.set_ylabel('|Convergence delta| (log scale)')
ax.set_title('LayerIG Convergence vs. Number of Steps\nFor BERT Token Attribution',
             fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('bert_ig_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_ig_convergence.png")
print("\nRecommendation: n_steps=50 provides good convergence for BERT.")

## 7. Attribution for Both Classes

Compute attributions for BOTH positive and negative class to show the full picture.

In [ ]:
# Compute attributions targeting each class separately
mixed_text = "A disappointingly average film with some bright moments."
ids, mask, ttype = tokenize_and_encode(mixed_text)
baseline_ids = make_pad_baseline(ids)
tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())

lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

all_class_attrs = []
for target_class in [0, 1]:  # 0=NEGATIVE, 1=POSITIVE
    attrs, delta = lig.attribute(
        inputs=ids, baselines=baseline_ids,
        additional_forward_args=(mask, ttype),
        target=target_class,
        n_steps=50, return_convergence_delta=True,
    )
    token_scores = attrs.sum(dim=-1).squeeze(0).detach().numpy()
    all_class_attrs.append(token_scores)

word_tokens_mixed, _ = aggregate_subwords(tokens, all_class_attrs[0])
_, word_attrs_neg = aggregate_subwords(tokens, all_class_attrs[0])
_, word_attrs_pos = aggregate_subwords(tokens, all_class_attrs[1])

# Net attribution: what discriminates positive from negative?
word_attrs_net = word_attrs_pos - word_attrs_neg

# Plot
fig, axes = plt.subplots(3, 1, figsize=(16, 9))

for ax, (label, attrs) in zip(axes, [
    (f'Attribution toward NEGATIVE class', word_attrs_neg),
    (f'Attribution toward POSITIVE class', word_attrs_pos),
    (f'Net attribution (POSITIVE - NEGATIVE)', word_attrs_net),
]):
    ax.axis('off')
    n = len(word_tokens_mixed)
    max_abs = np.abs(attrs).max() if np.abs(attrs).max() > 0 else 1
    attrs_norm = attrs / max_abs

    for i, (word, a, a_raw) in enumerate(zip(word_tokens_mixed, attrs_norm, attrs)):
        x = (i + 0.5) / n
        w = 1.0 / max(n, 1)
        if a > 0:
            r, g, b = 1-a*0.6, 1.0, 1-a*0.6
        else:
            r, g, b = 1.0, 1+a*0.6, 1+a*0.6
        ax.add_patch(plt.Rectangle((x-w*0.44, 0.2), w*0.88, 0.65,
                     facecolor=(np.clip(r,0,1), np.clip(g,0,1), np.clip(b,0,1)),
                     edgecolor='#dddddd', linewidth=0.4))
        ax.text(x, 0.55, word, ha='center', va='center',
                fontsize=10, color='black')
        ax.text(x, 0.12, f"{a_raw:+.2f}",
                ha='center', va='center', fontsize=6, color='#666666')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(label, fontsize=10, fontweight='bold')

plt.suptitle(f'Dual-Class Attribution: "{mixed_text}"',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_ig_dual_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_ig_dual_class.png")

print(f"\nTop positive discriminators (net attribution):")
for word, attr in sorted(zip(word_tokens_mixed, word_attrs_net), key=lambda x: -x[1])[:5]:
    print(f"  {word:<15}: {attr:+.4f}")
print(f"\nTop negative discriminators:")
for word, attr in sorted(zip(word_tokens_mixed, word_attrs_net), key=lambda x: x[1])[:5]:
    print(f"  {word:<15}: {attr:+.4f}")

## Self-Check Questions

1. **Baseline check:** Run the baseline through the model: `predict(tokenizer.decode(make_pad_baseline(ids)[0]))`. Is the prediction near 50% confidence? What does this tell you about the PAD baseline quality?

2. **Negation test:** For the sentence "The movie was NOT boring at all," which word gets the highest attribution? Does this make semantic sense? Compare with the sentence "The movie was boring" — how do the attributions change?

3. **Convergence:** We saw convergence delta improves with more steps. At n_steps=10, the delta is large. Does this mean the attribution is wrong, or just less precise? How would you decide if you need n_steps=100?

4. **Subword tokens:** Find a word in your test sentences that gets split into multiple subword tokens (check `result['tokens']`). What would happen if you reported the subword attributions separately instead of aggregating them?

**Exercise:** Modify `forward_func` to return only the logit for a specific class (e.g., `return model(...).logits[:, 1]`). Does this change the attribution output compared to using `target=1` in `lig.attribute()`? Why or why not?